In [306]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException   
from datetime import datetime, timedelta
import traceback
import pandas as pd
import numpy as np
from time import sleep
import urllib.parse
import json
import os
import requests
from bs4 import BeautifulSoup

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 30)
pd.set_option('future.no_silent_downcasting', True)

In [307]:
# DRIVER SETUP FUNCTIONS
def print_it(func):
    def wrapper(*args, **kwargs):
        print(f'Running {func.__name__}')
        return func(*args, **kwargs)
    return wrapper


def time_it(func):
    def wrapper(*args, **kwargs):
        # print(f'    Running {func.__name__}')
        start_time = datetime.now()
        result = func(*args, **kwargs)
        end_time = datetime.now()
        execution_time = (end_time - start_time).total_seconds()
        print(f'{func.__name__} --- {execution_time:.4f} sec')
        return result
    return wrapper


def reject_cookies(driver, temp=0, retries=2):
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.ID, 'onetrust-reject-all-handler')))
        driver.find_element(By.ID, 'onetrust-reject-all-handler').click()
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            # print(f'{retries} attempts failed to Reject cookies')
            print('Failed to reject cookies')
        else:
            reject_cookies(driver, temp + 1)


def skip_tutorial(driver, temp=0, retries=2):
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip')))
        driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()
        driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()
        driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            # print(f'{temp + 1} attempts failed to Skip the tutorial')
            print('Failed to skip the tutorial')
        else:
            skip_tutorial(driver, temp + 1)


def setup_driver(url):
    options = webdriver.ChromeOptions()
    options.set_capability("goog:loggingPrefs", {"performance": "INFO"})
    prefs = {"profile.managed_default_content_settings.images": 2}
    options.add_experimental_option("prefs", prefs)
    options.add_argument("--no-sandbox")
    options.add_argument("--headless")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-extensions")
    options.add_argument('--ignore-certificate-errors')
    options.add_argument('--disable-gpu')
    options.add_argument('--disable-dev-shm-usage')
    driver = webdriver.Chrome(options=options)

    driver.get(url)

    reject_cookies(driver)
    skip_tutorial(driver)

    return driver


def get_explorer_urls(product_id=None, category_id=None, object_id=None):
    # gets the corresponding urls to wallapop listing explorer
    base = 'https://es.wallapop.com/app/search?'
    urls = []

    if product_id:
        pass ## Do later
        # url = url if (product_id == None) else url + f'keywords={urllib.parse.quote(product_id)}&'
    else:
        url = base + f'object_type_ids={object_id}&filters_source=quick_filters&latitude=40.41956&longitude=-3.69196&'
        start, increment, loops = 0, 1, 11
        current_range = increment

        for i in range(loops):
            min = start
            max = start + current_range

            if i == 0:
                urls.append((f'{min}-{max}',(url + f'max_sale_price={max}')))
            elif i + 1 == len(range(loops)):
                urls.append((f'{min+.01}-{max}',(url + f'min_sale_price={min+.01}')))
            else:
                urls.append((f'{min+.01}-{max}',(url + f'min_sale_price={min+.01}&max_sale_price={max}')))

            start += current_range
            current_range += increment + round((i-1)**2) # Ramps up range ~according to previous scrapes

    '''Calculate a dynamic way to search space later

    from scipy.stats import norm
    import numpy as np

    # Desired percentage (e.g., 95%)
    percentage = 0.05

    # Calculate the z-score in the log-transformed space
    z_score_log = norm.ppf(percentage, loc=3.4, scale=0.67)

    # Transform back to the original scale
    z_score_original = np.power(10, z_score_log)
    print(z_score_log)
    print(f"The z-score for {percentage*100}% in the original scale is {z_score_original}")'''

    return urls


def load_more(driver, temp=0, retries=1):
    # Clicks button that initiates infinite scroll dynamic listings
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.ID, 'btn-load-more')))
        driver.find_element(By.ID, 'btn-load-more').click()
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            # print(f'{temp + 1} attempts failed to Click the load button')
            # print('Failed to click the load button')
            return True
        else:
            return load_more(driver, temp + 1)


def wait_listings(driver, temp=0, retries=1):
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'a.ItemCardList__item')))
        return True
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            # print(f'{temp + 1} attempts failed to Find listings')
            print('Failed to find listings')
        else:
            wait_listings(driver, temp + 1)


def convert_timestamp(ts):
    return datetime.fromtimestamp(ts / 1000).strftime('%y%m%d%H%M%S')


In [308]:
# CATEGORY FUNCTIONS
def get_categories(categories=False, category_dicts=None):
    if category_dicts is None:
        category_dicts = []
    if not categories:
        categories = requests.get('https://api.wallapop.com/api/v3/categories')
        categories = categories.json()['categories']
        return get_categories(categories=categories)
    else:
        for category in categories:
            if category['subcategories']:
                formatted_category = {key: [value] for key, value in category.items() if key != 'subcategories'}
                category_dicts.append(pd.DataFrame(formatted_category))
                category_dicts = get_categories(categories=category['subcategories'], category_dicts=category_dicts)
            else:
                formatted_category = {key: [value] for key, value in category.items()}
                category_dicts.append(pd.DataFrame(formatted_category))
        return category_dicts


def find_senior_parent_id(id, parent_map):
    if pd.isna(parent_map[id]):
        return id
    else:
        return find_senior_parent_id(parent_map[id], parent_map)


def build_path(id, parent_map):
    if pd.isna(parent_map[id]):
        return str(id)
    else:
        return build_path(parent_map[id], parent_map) + '/' + str(id)


def create_category_df():
    category_df = pd.concat(get_categories()).drop(['subcategories', 'presentation', 'category_leaf_selection_mandatory', 'icon', 'vertical_id'], axis=1)
    parent_map = category_df.set_index('id')['parent_id'].to_dict()
    category_df['senior_parent_id'] = category_df['id'].apply(lambda x: find_senior_parent_id(x, parent_map))
    category_df['senior_parent_id'] = category_df['senior_parent_id'].astype(int)
    category_df['path'] = category_df['id'].apply(lambda x: build_path(x, parent_map))
    category_df['path'] = category_df['path'].str.replace('.0', '', regex=False)
    category_df['attributes'] = category_df['attributes'].apply(json.dumps)
    category_df['attributes'] = category_df['attributes'].replace('{}', np.nan)
    category_df['attributes'] = category_df['attributes'].replace('{"excluded": []}', np.nan)
    return category_df.set_index('id').reset_index()


def setup_categories(category_ids):
    try:
        category_path = 'data/categories.csv'
        category_df = create_category_df()
        category_file = pd.read_csv(category_path)
        pd.testing.assert_frame_equal(category_df, category_file)
    except FileNotFoundError:
        category_file = category_df
        verify_filepath(category_path)
        category_df.to_csv(category_path, index=False)

    category_df_ids = category_file[category_file['parent_id'].isna()]
    category_ids = [id for id in category_ids if id in category_df_ids['id'].to_list()]

    bad_categories = []

    return category_file, category_ids


def save_attributes(object_type_ids):
    
    for _,row in object_type_ids.iterrows():
        if not pd.isna(row['attributes']):
            attribute_path = f'data/{row['path']}/{row['id']}_attributes.json'
            verify_filepath(attribute_path)
            with open(attribute_path, 'w') as json_file:
                json_file.write(row['attributes'])
    


In [309]:
# # DESCRIPTION PARSING FUNCTIONS
# import emoji
# from nltk.corpus import stopwords
# from nltk.tokenize import word_tokenize, sent_tokenize
# from langdetect import detect
# from langdetect.lang_detect_exception import LangDetectException


# LANGUAGE_MAP = {
#     'en': 'english',
#     'es': 'spanish',
#     'fr': 'french',
#     'de': 'german',
#     'it': 'italian',
#     'pt': 'portuguese',
#     'nl': 'dutch',
#     'ru': 'russian',
#     'zh-cn': 'chinese',
#     'ja': 'japanese',
# }


# def detect_language(text):
#     if text:
#         try:
#             return detect(text)
#         except LangDetectException:
#             return 'es'
#     return 'none'


# def calculate_char_metrics(text):
#     if text:
#         lowercase_char_count = 0
#         capitalized_char_count = 0
#         digit_count = 0
#         emoji_count = 0
#         non_alphanumeric_char_count = 0
#         line_break_count = 0
#         whitespace_count = 0
        
#         for char in text:
#             if char.islower():
#                 lowercase_char_count += 1
#             if char.isupper():
#                 capitalized_char_count += 1
#             if char.isdigit():
#                 digit_count += 1

#             if char in emoji.EMOJI_DATA:
#                 emoji_count += 1
#             elif not char.isalnum() and not char.isspace():
#                 non_alphanumeric_char_count += 1

#             if char == '\n':
#                 line_break_count += 1
#             elif char.isspace():
#                 whitespace_count += 1

#         metrics = {
#         'lowercase_char_count': lowercase_char_count,
#         'capitalized_char_count': capitalized_char_count,
#         'digit_count': digit_count,
#         'emoji_count': emoji_count,
#         'non_alphanumeric_char_count': non_alphanumeric_char_count,
#         'line_break_count': line_break_count,
#         'whitespace_count': whitespace_count,
#         }
#         return metrics
#     else:
#         return {
#         'lowercase_char_count': 0,
#         'capitalized_char_count': 0,
#         'digit_count': 0,
#         'emoji_count': 0,
#         'non_alphanumeric_char_count': 0,
#         'line_break_count': 0,
#         'whitespace_count': 0,
#         }


# def calculate_metrics(text, lang_code):
#     if text:
#         language = LANGUAGE_MAP.get(lang_code, 'spanish')
#         sentences = sent_tokenize(text)
#         words = [word for word in word_tokenize(text) if word.isalpha()]
#         stop_words = set(stopwords.words(language))
        
#         allcaps_word_count = 0
#         total_word_length = 0
#         unique_words = set()
#         stop_word_count = 0
        
#         for word in words:
#             if word.isupper():
#                 allcaps_word_count += 1
#             total_word_length += len(word)
#             unique_words.add(word)
#             if word.lower() in stop_words:
#                 stop_word_count += 1

#         avg_sent_len = round((len(words) / len(sentences) if sentences else 0), 2)
#         avg_word_len = round((total_word_length / len(words) if len(words) else 0), 2)
        
#         metrics = {
#             'sentence_count': len(sentences),
#             'avg_sent_len_cents': int(avg_sent_len*100),
#             'allcaps_word_count': allcaps_word_count,
#             'word_count': len(words),
#             'avg_word_len_cents': int(avg_word_len*100),
#             'unique_word_count': len(unique_words),
#             'stop_word_count': stop_word_count,
#         }
#         return metrics
#     else:
#         return {
#             'sentence_count': 0,
#             'avg_sent_len_cents': 0,
#             'allcaps_word_count': 0,
#             'word_count': 0,
#             'avg_word_len_cents': 0,
#             'unique_word_count': 0,
#             'stop_word_count': 0,
#         }

In [310]:
# SCRAPING FUNCTIONS

def verify_filepath(filepath):
    directory = os.path.dirname(filepath)
    if not os.path.exists(directory):
        os.makedirs(directory)


def save_df(filepath, df):
    assert len(df) > 0
    df = df.dropna(how='all').drop_duplicates()
    df.to_csv(filepath, index=False)


def load_dfs(category_id=None, object_id=None):
    global category_df
    dfs = []
    path = category_df.loc[category_df['id'] == object_id, 'path'].values[0]
    directory = f'data/{path}/'
    for filename in os.listdir(directory):
        if filename.startswith(f"{category_id}"):
            filepath = os.path.join(directory, filename)
            df = pd.read_csv(filepath)
            dfs.append(df)
    object_df = pd.concat(dfs)
    return object_df


def check_redundancy(json_df, object_df):
    global len_scraped
    json_df_comp = json_df.drop(columns=['date_last_scraped', 'date_first_seen_reserved'], errors='ignore')
    object_df_comp = object_df.drop(columns=['date_last_scraped', 'date_first_seen_reserved'], errors='ignore')
    json_df_comp = json_df_comp[object_df_comp.columns]
    if json_df_comp.apply(tuple, axis=1).isin(object_df_comp.apply(tuple, axis=1)).all():
        len_scraped += len(json_df)
        return True


def find_matches(object_df, json_df):
    global new_items
    global modified_items
    for _, row in json_df.iterrows():
        if row['id'] in object_df['id'].values:
            idx = object_df.index[object_df['id'] == row['id']].tolist()[0]
            modified_columns = ['date_first_seen_reserved', 'date_last_modified']
            if not row[modified_columns].equals(object_df.loc[idx, modified_columns]):
                modified_items += 1
                common_columns = object_df.columns.intersection(json_df.columns)
                differences = {col: (object_df.at[idx, col], row[col]) for col in common_columns if object_df.at[idx, col] != row[col]}

                if object_df.at[idx, 'date_first_seen_reserved'] != 0:
                    try:
                        del differences['date_first_seen_reserved']
                    except KeyError:
                        pass
                for col, value in differences.items():
                    object_df.at[idx, col] = value[1]
                    
            else:
                object_df.loc[idx, 'date_last_scraped'] = row['date_last_scraped']
        else:
            object_df = pd.concat([object_df, pd.DataFrame([row])], ignore_index=True)
            new_items += 1

    return object_df


def compare_scraping(json_df, category_id=None, object_id=None, range_name=None, all=False):
    global len_scraped
    global category_df
    if range_name:
        try:
            path = category_df.loc[category_df['id'] == object_id, 'path'].values[0]
            filepath = f'data/{path}/{category_id}_{object_id}_price:{range_name}.csv'
            object_df = pd.read_csv(filepath)
            if not all and check_redundancy(json_df, object_df):
                return json_df, 0
            
            object_df = find_matches(object_df, json_df)          
            save_df(filepath, object_df)

        except FileNotFoundError:
            verify_filepath(filepath)
            save_df(filepath, json_df)

        len_scraped += len(json_df)
        return json_df, 1
        
    else: # for newest
        if not all:
            object_df = load_dfs(category_id=category_id, object_id=object_id)
            if check_redundancy(json_df, object_df):
                return json_df, 0
        path = category_df.loc[category_df['id'] == object_id, 'path'].values[0]
        directory = f'data/{path}/'
        urls = pd.read_csv(f'{directory}{object_id}_urls.csv')
        range_names = urls['range_name'].to_list()
        for i, range_name in enumerate(range_names):
            min_price, max_price = map(float, range_name.split('-'))
            min_price = int(min_price*100)
            max_price = int(max_price*100)
            
            if i+1 == len(range_names):
                df_splice = json_df[(json_df['price_cents'] >= min_price)]
            else:
                df_splice = json_df[(json_df['price_cents'] >= min_price) & (json_df['price_cents'] <= max_price)]
                
            if df_splice.empty:
                continue
            else:
                try:
                    filepath = f'{directory}{category_id}_{object_id}_price:{range_name}.csv'
                    object_df = pd.read_csv(filepath)
                    object_df = find_matches(object_df, df_splice)
                    save_df(filepath, object_df)
                except FileNotFoundError:
                    verify_filepath(filepath)
                    save_df(filepath, json_df)

        len_scraped += len(json_df)
        return json_df, 1


def parse_json(response_json):
    df = pd.DataFrame(response_json['data']['section']['payload']['items'])

    # Split up shipping
    shipping_normalized = pd.json_normalize(df['shipping'])
    df = df.reset_index(drop=True)
    shipping_normalized = shipping_normalized.reset_index(drop=True)
    df = pd.concat([df, shipping_normalized], axis=1)

    # Clean up othe metrics
    df['country_code'] = df['location'].apply(lambda x: x['country_code'])
    df['num_images'] = df['images'].apply(lambda x:len(x))
    df['refurbished'] = df['is_refurbished'].apply(lambda x: list(x.values())[0])
    df['reserved'] = df['reserved'].apply(lambda x: list(x.values())[0])
    df['currency'] = df['price'].apply(lambda x: list(x.values())[1])
    df['price_cents'] = df['price'].apply(lambda x: int(list(x.values())[0]*100))
    df['object_id'] = df['taxonomy'].apply(lambda x: int(x[-1]['id']))

    replace_dict = {'none': 0,
                    True: 1,
                    False: 0}
    df['refurbished'] = df['refurbished'].replace(replace_dict).astype(int)
    df['reserved'] = df['reserved'].replace(replace_dict).astype(int)
    df['user_allows_shipping'] = df['user_allows_shipping'].replace(replace_dict).astype(int)
    df['item_is_shippable'] = df['item_is_shippable'].replace(replace_dict).astype(int)

    # Get time metrics
    current_time = int(datetime.now().strftime('%y%m%d%H%M%S'))
    df['date_first_seen_reserved'] = df['reserved'].apply(lambda x: current_time if x == 1 else 0)
    df['date_last_modified'] = df['modified_at'].apply(lambda ts: int(datetime.fromtimestamp(ts / 1000).strftime('%y%m%d%H%M%S')))
    df['date_first_created'] = df['created_at'].apply(lambda ts: int(datetime.fromtimestamp(ts / 1000).strftime('%y%m%d%H%M%S')))
    df['date_last_scraped'] = current_time

    df = df.drop(['images', 'shipping', 'price', 'discount', 'favorited', 'cost_configuration_id', 'taxonomy', 'location', 'favoriteable', 'created_at', 'modified_at', 'is_refurbished', 'is_favoriteable', 'category_id', 'bump'], axis=1, errors='ignore')

    # # Get language and word metrics
    # df['lang_code'] = df['description'].apply(detect_language)
    # metrics_df = df.apply(lambda row: calculate_metrics(row['description'], row['lang_code']), axis=1).apply(pd.Series)
    # metrics_df = metrics_df.astype({
    #     'allcaps_word_count': 'int',
    #     'word_count': 'int',
    #     'unique_word_count': 'int',
    #     'stop_word_count': 'int',
    #     'sentence_count': 'int',
    # })
    # df = pd.concat([df, metrics_df], axis=1)

    # # Get char metrics
    # metrics_df = df['description'].apply(calculate_char_metrics).apply(pd.Series)
    # df = pd.concat([df, metrics_df], axis=1)

    # df = df.drop(['images', 'shipping', 'price', 'discount', 'favorited', 'cost_configuration_id', 'taxonomy', 'location', 'favoriteable', 'created_at', 'modified_at', 'is_refurbished', 'is_favoriteable', 'category_id', 'bump', 'description'], axis=1, errors='ignore') # For when we activate description parsing

    return df


def get_response_body(driver, request_id, temp=0, retries=30):
    try:
        response_body = driver.execute_cdp_cmd("Network.getResponseBody", {"requestId": request_id})
        return response_body
    except Exception:
        if temp >= retries:
            print(f'++++++++++++++ {temp} attemps failed to get json response body')
        else:
            sleep(temp*temp/1000)
            return get_response_body(driver, request_id, temp + 1)


def get_scraping(driver, timed_out=None, timer=None, category_id=None, object_id=None, range_name=None, all=False, first=False):
    scrapings_df = pd.DataFrame()
    new_counts = 0
    
    logs = driver.get_log("performance")
    while logs:
        for log in logs:
            log_json = json.loads(log["message"])["message"]
            if log_json["method"] == 'Network.requestWillBeSent':
                try:
                    response_url = log_json["params"]["documentURL"]
                    if response_url.startswith("https://api.wallapop.com/api/v3/search?"):
                        request_id = log_json["params"]["initiator"]["requestId"]
                        try:
                            response_body = get_response_body(driver, request_id)
                            if not response_body or 'body' not in response_body:
                                continue  # Skip to the next log entry if the response is empty or invalid
                            else:
                                response_json = json.loads(response_body['body'])
                            if response_json['data']['section']['payload']['items']:
                                try:
                                    json_df = parse_json(response_json)
                                    json_df, new_count = compare_scraping(json_df, category_id=category_id, object_id=object_id, range_name=range_name, all=all)
                                    new_counts += new_count
                                    # print(f'redundant: {new_counts}')
                                    scrapings_df = pd.concat([scrapings_df, json_df])
                                except KeyError as e:
                                    print(e)
                                    print(response_json)

                        except Exception as e:
                            print(f'GET_SCRAPING ERROR: {e}')
                            # print('\n')
                            # print(log_json)
                            # print('\n')
                            traceback.print_exc()
                            print('\n')

                except KeyError:
                    print('KEYERROR IN GET SCRAPING')
                    print(f'\n\n{log_json}')
                    
        logs = driver.get_log("performance")

    # # Print stats
    # timespent = datetime.now() - timer
    # efficiency = round(len(scrapings_df)/timespent.total_seconds(), 3)
    # print(f'    time: {timespent}')
    # print(f'    count: {len(scrapings_df)}')
    # print(f'    efficiency = {efficiency}/sec')
        
    if timed_out:
        # print('    timed_out scrape')
        return True
    elif not new_counts:
        # print('    redundant scrape')
        return True
    elif first:
        return False
    else:
        clear_explorer(driver)

        return scroll_explorer(driver, category_id=category_id, object_id=object_id, range_name=range_name, all=all)


def scroll_explorer(driver, category_id=None, object_id=None, range_name=None, all=False):
    global max_scrolls
    previous_height = driver.execute_script("return document.body.scrollHeight")
    sleep_time = 0.1
    count = 0
    timer = datetime.now()
    complete = False

    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        if count == max_scrolls:
            # print(f'{object_id} Price range:{range_name} max_scrolls count reached')
            complete = get_scraping(driver, timed_out=False, timer=timer, category_id=category_id, object_id=object_id, range_name=range_name, all=all)
        try:
            WebDriverWait(driver, sleep_time).until(lambda driver: driver.execute_script("return document.body.scrollHeight") != previous_height)
            count += 1
            sleep_time /= 2
            previous_height = driver.execute_script("return document.body.scrollHeight")
        except TimeoutException:
            sleep_time += sleep_time
            if sleep_time > 1.5:
                # print(f'{object_id} {range_name} Scroll: Timed out')
                complete = get_scraping(driver, timed_out=True, timer=timer, category_id=category_id, object_id=object_id, range_name=range_name, all=all)

        if complete:
            return True


def clear_explorer(driver):
    driver.execute_script("window.scrollTo(0, 0);")
    try:
        # Find elements and delete all siblings
        element = driver.find_element(By.CSS_SELECTOR, 'a.ItemCardList__item')
        parent = element.find_element(By.XPATH, '..')
        siblings = parent.find_elements(By.XPATH, '*')
        for sibling in siblings:
            driver.execute_script("arguments[0].remove();", sibling)
    except NoSuchElementException:
        print('    No listings found to clear to clear')


def start_scraping(urls_df, urls_path, product_id=None, category_id=None, object_id=None):
    global len_scraped
    global category_df
    object_count = 0

    time_threshold = 24*7 ##############

    time_ago = int((datetime.now() - timedelta(hours=time_threshold)).strftime('%y%m%d%H'))
    pending_urls = urls_df[urls_df['last_checked'] < time_ago]

    for _, row in pending_urls.iterrows():
        range_start = datetime.now()
        len_scraped = 0
        range_name = row['range_name']
        url = row['url']
        complete = False
        path = category_df.loc[category_df['id'] == object_id, 'path'].values[0]
        category_name = category_df.loc[category_df['id'] == object_id, 'name'].values[0]
        print(f'\n{path} --- {category_name} --- Price range:{range_name} --- Checking low-to-high')
        
        driver = setup_driver(url + '&order_by=price_low_to_high')
        if wait_listings(driver):
            try:
                load_more(driver)
                complete = scroll_explorer(driver, category_id=category_id, object_id=object_id, range_name=range_name, all=True)
                driver.quit()
            except Exception as e:
                # print('\n')
                # traceback.print_exc()
                # print('\n')
                driver.quit()
                print(f'Low-High ERROR: {e}')

            if not complete:
                print(f'{path} --- Price range:{range_name} --- Checking high-to-low')
                driver = setup_driver(url + '&order_by=price_high_to_low')
                if wait_listings(driver):
                    try:
                        load_more(driver)
                        complete = scroll_explorer(driver, category_id=category_id, object_id=object_id, range_name=range_name)
                        driver.quit()
                    except Exception as e:
                        # print('\n')
                        # traceback.print_exc()
                        # print('\n')
                        driver.quit()
                        print(f'High-Low ERROR: {e}')

                    if not complete:
                        print(f'{path} --- Price range:{range_name} --- Checking closest')
                        driver = setup_driver(url + '&order_by=closest')
                        if wait_listings(driver):
                            try:
                                load_more(driver)
                                complete = scroll_explorer(driver, category_id=category_id, object_id=object_id, range_name=range_name)
                                driver.quit()
                            except Exception as e:
                                # print('\n')
                                # traceback.print_exc()
                                # print('\n')
                                driver.quit()
                                print(f'Closest ERROR: {e}')

                            if not complete:
                                print(f'{path} --- Price range:{range_name} --- Checking newest')
                                driver = setup_driver(url + '&order_by=newest')
                                if wait_listings(driver):
                                    try:
                                        load_more(driver)
                                        complete = scroll_explorer(driver, category_id=category_id, object_id=object_id, range_name=range_name)
                                        driver.quit()
                                    except Exception as e:
                                        # print('\n')
                                        # traceback.print_exc()
                                        # print('\n')
                                        driver.quit()
                                        print(f'Newest ERROR: {e}')
                                else:
                                    driver.quit()
                        else:
                            driver.quit()
                else:
                    driver.quit()
        else:
            driver.quit()

        # Update categories.csv
        urls_df.loc[urls_df['range_name'] == range_name, 'last_checked'] = int(datetime.now().strftime('%y%m%d%H%M%S'))
        urls_df.to_csv(urls_path, index=False)

        timespent = datetime.now() - range_start
        object_count += len_scraped
        # print(f'\n{object_id} --- Price range:{range_name} --- Checked')
        print(f'    {range_name} took: {timespent}')
        print(f'    {range_name} scraped: {len_scraped}')
        print(f'    {range_name} efficency: {round(len_scraped/timespent.total_seconds(), 3)}/sec')


    return object_count


def check_categories(category_ids=[24200, 12579, 13100], product_id=None, reverse=False):
    global category_df

    category_df, category_ids = setup_categories(category_ids)

    for category_id in category_ids:
        object_type_ids = category_df[category_df['senior_parent_id'] == category_id].sort_values(by='id', ascending=reverse)
        save_attributes(object_type_ids)
        object_type_ids = object_type_ids[~object_type_ids['id'].isin(object_type_ids['parent_id'])] # Select leaves of category tree

        for object_id in object_type_ids['id']:
            object_start = datetime.now()
            object_name = category_df.loc[category_df['id'] == object_id, 'name'].values[0]
            path = category_df.loc[category_df['id'] == object_id, 'path'].values[0]
            # print(f'\n{path} CHECKING {object_name}')

            urls = get_explorer_urls(product_id=product_id, category_id=category_id, object_id=object_id)
            try:
                urls_path = f'data/{path}/{object_id}_urls.csv'
                urls_df = pd.read_csv(urls_path)
            except FileNotFoundError:
                urls_df = pd.DataFrame(urls)
                urls_df['last_checked'] = 0
                urls_df = urls_df.rename(columns={0:'range_name', 1:'url'})
                verify_filepath(urls_path)
                urls_df.to_csv(urls_path)

            try:
                object_count = start_scraping(urls_df, urls_path, product_id=product_id, category_id=category_id, object_id=object_id)
            except Exception as e:
                print('\n')
                print(f'ERROR: {e}')
                traceback.print_exc()
                print('\n')

            timespent = datetime.now() - object_start
            print(f'\n+++  {path} {object_name} OBJECT CHECKED')
            print(f'+++     {object_name} took: {timespent}')
            print(f'+++     {object_name} scraped: {object_count}')
            print(f'+++     {object_name} efficency: {round(object_count/timespent.total_seconds(), 3)}/sec')

    print('\n=== All categories checked ===\n')


def check_newest(category_ids=[24200, 12579, 13100], product_id=None, reverse=False):
    global len_scraped
    global category_df
    global new_items
    global modified_items
    
    category_df, category_ids = setup_categories(category_ids)

    for category_id in category_ids:
        object_type_ids = category_df[category_df['senior_parent_id'] == category_id].sort_values(by='id', ascending=reverse)
        save_attributes(object_type_ids)
        object_type_ids = object_type_ids[~object_type_ids['id'].isin(object_type_ids['parent_id'])]

        for object_id in object_type_ids['id']:
            new_items = 0
            modified_items = 0
            len_scraped = 0
            object_start = datetime.now()
            object_name = category_df.loc[category_df['id'] == object_id, 'name'].values[0]
            path = category_df.loc[category_df['id'] == object_id, 'path'].values[0]
            print(f'\n{path} {object_name}')

            url = f'https://es.wallapop.com/app/search?object_type_ids={object_id}&filters_source=quick_filters&latitude=40.41956&longitude=-3.69196&order_by=newest'            
            driver = setup_driver(url)
            if wait_listings(driver):
                try:
                    complete = get_scraping(driver, category_id=category_id, object_id=object_id, first=True)
                    if not complete:
                        load_more(driver)
                        scroll_explorer(driver, category_id=category_id, object_id=object_id)
                        driver.quit()
                except Exception as e:
                    # print('\n')
                    # traceback.print_exc()
                    # print('\n')
                    driver.quit()
                    # print(f'{object_id} = Price range:{range_name} = newest ERROR: {e}')
                    print(f'ERROR: {e}')
            else:
                driver.quit()

            timespent = datetime.now() - object_start
            # print(f'    {object_id} {object_name}')
            print(f'    {object_name} took: {timespent}')
            print(f'    {object_name} scraped: {len_scraped}')
            print(f'    {object_name} new items: {new_items}')
            print(f'    {object_name} modified items: {modified_items}')
            print(f'    {object_name} efficency: {round(len_scraped/timespent.total_seconds(), 3)}/sec')

            metadata = pd.DataFrame({'time': int(datetime.now().strftime('%y%m%d%H%M%S')),
                                     'new_items': new_items,
                                     'modified_items': modified_items}, index=[0])
            try:
                meta_path = f'data/{path}/{object_id}_metadata.csv'
                meta_df = pd.read_csv(meta_path)
                meta_df = pd.concat([meta_df, metadata], ignore_index=True)
                meta_df.to_csv(meta_path, index=False)
            except FileNotFoundError:
                verify_filepath(meta_path)
                metadata.to_csv(meta_path, index=False)

    print('\n=== All categories checked ===\n')



In [311]:
# ITEM FUNCTIONS
def convert_timestamp(ts):
    return datetime.fromtimestamp(ts / 1000).strftime('%y%m%d%H%M%S')


def update_listing(listing_df, full_object_df):
    idx = full_object_df.index[full_object_df['id'] == listing_df['id'].values[0]].tolist()[0]

    replace_dict = {'none': 0,
                    'None': 0,
                    None: 0,
                    True: 1,
                    False: 0}

    full_object_df.at[idx, 'views'] = int(listing_df['views'].values[0])
    full_object_df.at[idx, 'favorites'] = int(listing_df['favorites'].values[0])
    full_object_df.at[idx, 'brand'] = listing_df['brand'].values[0]
    full_object_df.at[idx, 'model'] = listing_df['model'].values[0]
    full_object_df.at[idx, 'condition'] = listing_df['condition_value'].values[0]
    full_object_df.at[idx, 'color'] = listing_df['goodsAndFashionInfo_color_value'].values[0]
    full_object_df.at[idx, 'characteristics'] = listing_df['characteristics'].values[0]
    full_object_df.at[idx, 'retail_cents'] = int(listing_df['retailPrice'].values[0]) * 100
    full_object_df.at[idx, 'financed_cents'] = int(listing_df['price_financed'].values[0]) * 100

    full_object_df.at[idx, 'bumped'] = listing_df['flags_bumped'].values[0].replace(replace_dict).astype(int)
    full_object_df.at[idx, 'favorited'] = listing_df['flags_favorited'].values[0].replace(replace_dict).astype(int)
    full_object_df.at[idx, 'expired'] = listing_df['flags_expired'].values[0].replace(replace_dict).astype(int)
    full_object_df.at[idx, 'sold'] = listing_df['flags_sold'].values[0].replace(replace_dict).astype(int)
    full_object_df.at[idx, 'on_hold'] = listing_df['flags_onHold'].values[0].replace(replace_dict).astype(int)
    full_object_df.at[idx, 'bulky'] = listing_df['isBulky'].values[0].replace(replace_dict).astype(int)
    
    full_object_df.at[idx, 'max_kg'] = listing_df['goodsAndFashionInfo_upToKg'].values[0]
    full_object_df.at[idx, 'measures'] = listing_df['measures'].values[0]
    full_object_df.at[idx, 'size'] = listing_df['goodsAndFashionInfo_size'].values[0]
    full_object_df.at[idx, 'stock'] = listing_df['stock'].values[0]
    full_object_df.at[idx, 'car_info'] = listing_df['carInfo'].values[0]
    full_object_df.at[idx, 'real_estate_info'] = listing_df['realEstateInfo'].values[0]
    full_object_df.at[idx, 'isbn'] = listing_df['isbn'].values[0]

    modified_date = convert_timestamp(listing_df['modifiedDate'].values[0])
    if full_object_df.iloc[0]['date_last_modified'] != modified_date:
        full_object_df.at[idx, 'date_last_modified'] = modified_date
        full_object_df.at[idx, 'price_cents'] = int(listing_df['price_cash_amount'].values[0]) * 100
        full_object_df.at[idx, 'currency'] = listing_df['price_cash_currency'].values[0]

        full_object_df.at[idx, 'reserved'] = listing_df['flags_reserved'].values[0].replace(replace_dict).astype(int)
        full_object_df.at[idx, 'item_is_shippable'] = listing_df['shipping_isItemShippable'].values[0].replace(replace_dict).astype(int)
        full_object_df.at[idx, 'user_allows_shipping'] = listing_df['shipping_isShippingAllowedByUser'].values[0].replace(replace_dict).astype(int)
        full_object_df.at[idx, 'refurbished'] = listing_df['isRefurbished'].values[0].replace(replace_dict).astype(int)
        
        image_columns = listing_df.filter(regex='^images').columns
        image_numbers = image_columns.str.extract(r'images_(\d+)_')[0].unique()
        full_object_df.at[idx, 'num_images'] = len(image_numbers)

        translated = listing_df['title_translated'].values[0]
        if translated:
            full_object_df.at[idx, 'title'] = translated
            full_object_df.at[idx, 'description'] = listing_df['description_translated'].values[0]
        else:
            full_object_df.at[idx, 'title'] = listing_df['title_original'].values[0]
            full_object_df.at[idx, 'description'] = listing_df['title_original'].values[0]

    return full_object_df


def flatten_dict(d, parent_key='', sep='_'):
    items = []
    for k, v in d.items():
        new_key = f'{parent_key}{sep}{k}' if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        elif isinstance(v, list):
            for i, item in enumerate(v):
                items.extend(flatten_dict({f'{new_key}_{i}': item}, '', sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)


def check_items(category_ids=[24200, 12579, 13100], product_id=None, reverse=False):
    global len_scraped
    global category_df
    
    category_df, category_ids = setup_categories(category_ids)

    for category_id in category_ids:
        object_type_ids = category_df[category_df['senior_parent_id'] == category_id].sort_values(by='id', ascending=reverse)
        save_attributes(object_type_ids)
        object_type_ids = object_type_ids[~object_type_ids['id'].isin(object_type_ids['parent_id'])]

        for object_id in object_type_ids['id']:
            len_scraped = 0
            object_start = datetime.now()
            object_name = category_df.loc[category_df['id'] == object_id, 'name'].values[0]
            path = category_df.loc[category_df['id'] == object_id, 'path'].values[0]
            print(f'\n{path} {object_name}')

            try:
                full_object_path = f'data/{path}/{object_id}_data.csv'
                full_object_df = pd.read_csv(full_object_path)
            except FileNotFoundError:
                full_object_df = load_dfs(category_id=category_id, object_id=object_id)

            for web_slug in full_object_df['web_slug'].to_list():
                url = 'https://es.wallapop.com/item/' + web_slug
                request = requests.get(url)
                soup = BeautifulSoup(request.text, 'html.parser')
                script_tag = soup.find('script', id='__NEXT_DATA__')
                json_content = script_tag.string
                parsed_json = json.loads(json_content)
                flattened_data = flatten_dict(parsed_json['props']['pageProps']['item'])
                listing_df = pd.DataFrame([flattened_data])
                full_object_df = update_listing(listing_df, full_object_df)


            timespent = datetime.now() - object_start
            # print(f'    {object_id} {object_name}')
            print(f'    {object_name} took: {timespent}')
            print(f'    {object_name} scraped: {len_scraped}')
            print(f'    {object_name} efficency: {round(len_scraped/timespent.total_seconds(), 3)}/sec')

    print('\n=== All categories checked ===\n')

In [325]:
url = 'https://es.wallapop.com/item/mono-bomberos-848615005'
url = 'https://es.wallapop.com/item/iphone-6s-1039028626'
request = requests.get(url)
soup = BeautifulSoup(request.text, 'html.parser')
script_tag = soup.find('script', id='__NEXT_DATA__')
json_content = script_tag.string
parsed_json = json.loads(json_content)
flattened_data = flatten_dict(parsed_json['props']['pageProps']['item'])
listings_df = pd.DataFrame([flattened_data])

In [ ]:
{'title': 'iphone 13',
          'description': 'Selling Iphone 13 256GB 2021',
          'price': 3000}
{'title': 'iphone 14max brand new',
          'description': "iphone 14max 256GB 2024. It's brand new",
          'price': 3500}
{'title': 'iphone 13 -- worn out -- value price',
          'description': 'iphone 13 with a few scratches which you can see in the images, but it runs perfectly well',
          'price': 2000}
{'title': 'acer swift 3',
          'description': "ACER Swift 3 from Spain. It's an old laptop but it runs great",
          'price': 2000}

In [316]:
full_object_df = pd.read_csv('data/18000/10072/18000_10072_price:14.01-25.csv')
dfs = []
for i, web_slug in enumerate(full_object_df['web_slug'].to_list()):
    now = datetime.now()
    url = 'https://es.wallapop.com/item/' + web_slug
    request = requests.get(url)
    if request.status_code != 200:
        print(f'bad request for {url}')
        print()
        print(f'{round(i/len(full_object_df), 5)} Time: {datetime.now() - now}')
        continue

    soup = BeautifulSoup(request.text[56500:], 'html.parser')
    script_tag = soup.find('script', id='__NEXT_DATA__')
    json_content = script_tag.string
    parsed_json = json.loads(json_content)
    try:
        flattened_data = flatten_dict(parsed_json['props']['pageProps']['item'])
    except Exception as e:
        print(f'ERROR {e}')
        traceback.print_exc()
        print(f'{round(i/len(full_object_df), 5)} Time: {datetime.now() - now}')
        continue
    listing_df = pd.DataFrame([flattened_data])
    dfs.append(listing_df)
    print(f'{round(i/len(full_object_df), 5)} Time: {datetime.now() - now}')
    # full_object_df = full_object_df.merge(listing_df, on='id', how='outer')
    # full_object_df = update_listing(listing_df, full_object_df)

# full_object_df

0.0 Time: 0:00:00.926024
2e-05 Time: 0:00:00.806934
4e-05 Time: 0:00:00.511984
6e-05 Time: 0:00:00.427049
7e-05 Time: 0:00:00.738283
9e-05 Time: 0:00:00.992677
0.00011 Time: 0:00:00.419305
0.00013 Time: 0:00:00.489246
0.00015 Time: 0:00:00.364316
0.00017 Time: 0:00:00.455627
0.00019 Time: 0:00:00.410871
0.00021 Time: 0:00:00.404239
0.00022 Time: 0:00:00.421110
0.00024 Time: 0:00:00.391748
0.00026 Time: 0:00:00.448077
0.00028 Time: 0:00:00.372820
0.0003 Time: 0:00:00.398371
0.00032 Time: 0:00:00.530590
0.00034 Time: 0:00:00.414475
0.00036 Time: 0:00:00.531754
0.00037 Time: 0:00:00.589621
0.00039 Time: 0:00:00.511190
0.00041 Time: 0:00:00.362608
0.00043 Time: 0:00:00.455775
0.00045 Time: 0:00:00.430098
0.00047 Time: 0:00:00.493726
0.00049 Time: 0:00:00.395179
0.0005 Time: 0:00:00.526548
0.00052 Time: 0:00:00.408087
0.00054 Time: 0:00:00.406138
0.00056 Time: 0:00:00.334054
0.00058 Time: 0:00:00.351681
0.0006 Time: 0:00:00.547050
0.00062 Time: 0:00:00.392341
0.00064 Time: 0:00:00.432176
0.

KeyboardInterrupt: 

In [49]:
# import os
# import pandas as pd

# def add_object_id_to_csvs(root_dir):
#     for subdir, _, files in os.walk(root_dir):
#         for file in files:
#             if file.endswith('.csv') and file.startswith('24200'):
#                 file_path = os.path.join(subdir, file)
#                 parent_dir = os.path.basename(subdir)
                
#                 df = pd.read_csv(file_path)
#                 df['object_id'] = parent_dir
#                 df.to_csv(file_path, index=False)
                
#                 print(f"Updated {file_path} with object_id = {parent_dir}")

# add_object_id_to_csvs('data/24200')